CHiamata API 

In [ ]:
from github import Github
import json

TOKEN = "..."

g = Github(TOKEN)

C:\Users\vitog\AppData\Local\Temp\ipykernel_6720\1708734020.py:6: DeprecationWarning: Argument login_or_token is deprecated, please use auth=github.Auth.Token(...) instead
  g = Github(TOKEN)


In [ ]:
import pandas as pd

dataset_path = 'C:\\dev\\SE4AI-base\\gigawork\\dataset.csv'

df = pd.read_csv(dataset_path)

In [ ]:
import requests

# a volte le repository possono essere state trasferite quindi bisogna risolvere i redirect per ottenere il nome completo corretto
http = requests.Session()
http.headers.update({
    "Authorization": f"Bearer {TOKEN}",
    "Accept": "application/vnd.github+json",
})

redirect_cache = {}

def resolve_owner_repo(owner_repo: str) -> str:
    if owner_repo in redirect_cache:
        return redirect_cache[owner_repo]

    url = f"https://api.github.com/repos/{owner_repo}"
    try:
        # allow_redirects=True segue i trasferimenti repo
        r = http.get(url, allow_redirects=True, timeout=20)
        if r.status_code == 200:
            full_name = r.json().get("full_name", owner_repo)
            redirect_cache[owner_repo] = full_name
            return full_name
    except Exception:
        pass

    redirect_cache[owner_repo] = owner_repo
    return owner_repo


In [ ]:
# ciclo su tutti i commithash nel dataset e recupero i dati delle esecuzioni per ogni commit
import time

results = []
errors = []

work = (
    df[["repository", "commit_hash"]]
    .dropna()
    .drop_duplicates()
)

for repository_key, group in work.groupby("repository"):
    original_owner_repo = repository_key.replace("__", "/", 1)
    resolved_owner_repo = resolve_owner_repo(original_owner_repo)

    try:
        repo = g.get_repo(resolved_owner_repo)

        for sha in group["commit_hash"]:
            try:
                runs = repo.get_workflow_runs(head_sha=sha)
                time.sleep(1)

                for run in runs:
                    raw = run.raw_data
                    raw["dataset_repository"] = repository_key
                    raw["dataset_commit_hash"] = sha
                    raw["original_owner_repo"] = original_owner_repo
                    raw["resolved_owner_repo"] = resolved_owner_repo
                    results.append(raw)

            except Exception as e:
                errors.append({
                    "repository": repository_key,
                    "original_owner_repo": original_owner_repo,
                    "resolved_owner_repo": resolved_owner_repo,
                    "commit_hash": sha,
                    "error": str(e),
                })
    except Exception as e:
        errors.append({
            "repository": repository_key,
            "original_owner_repo": original_owner_repo,
            "resolved_owner_repo": resolved_owner_repo,
            "commit_hash": None,
            "error": str(e),
        })

print(f"Workflow runs raccolte: {len(results)}")
print(f"Errori: {len(errors)}")

In [ ]:
with open("C:\\dev\\SE4AI-base\\github_api_results\\workflow_runs_full_data.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

with open("C:\\dev\\SE4AI-base\\github_api_results\\workflow_runs_errors.json", "w", encoding="utf-8") as f:
    json.dump(errors, f, indent=2, ensure_ascii=False)

In [5]:
# path, name, display_title, head_sha, conclusion, status, event, head_commit.message
repo = "AntonOsika/gpt-engineer"

repo = g.get_repo("AntonOsika/gpt-engineer")
sha = "9cc9cf732918d2bbb653074be8e26040acbc4367"

runs = repo.get_workflow_runs(head_sha=sha)

print(f"Workflow runs per commit {sha}: {runs.totalCount}")


Workflow runs per commit 9cc9cf732918d2bbb653074be8e26040acbc4367: 0
